In [27]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [28]:
input_data = "example_input_files/relationship_extraction_0081.csv"
output_data = "example_output_files/relationship_extraction_0081.xlsx"

In [43]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your open ai api"

## common functions

In [30]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [31]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [32]:
"""
    This function is different a bit among different tasks, here is for the relationship extraction task
"""

def construct_annotation(df, results, task_name="semantic_matching",rule_id="0081"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
        the options of rule_id are 0081, 0001, and 0145
            if rule_id is 0081, relation_type is Reversal
            if rule_id is 0001, relation_type is CombinationErr
            if rule_id is 0145, relation_type is Inappropriateness
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []

    if rule_id == "0081":
        relation_type = "Reversal"
    elif rule_id == "0001":
        relation_type = "CombinationErr"
    else:
        relation_type = "Inappropriateness"

    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"second_round_filter/segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## 
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        
        answers = [(tag.get("tag1"),tag.get("tag2")) for tag in res.get("error_tags")] ##special for relationship extraction task
        flattened_answers = [x for pair in answers for x in pair] ##special for relationship extraction task, which is easier to check if the tag is in the specific document

        # convert the answers to triplet format
        answers = [(pair[0],relation_type,pair[1]) for pair in answers]
        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True

            for ans in flattened_answers:
                ## check the calculation linkbase
                if ans in temp_cal:
                    continue
                elif ans.replace(":","") in temp_cal:
                    continue
                elif ans.replace(":","_") in temp_cal:
                    continue
                else:
                    is_appear = False
                    break
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for relationship extraction task

In [33]:
re_81_df = pd.read_csv(input_data)

In [34]:
re_81_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0081.9278,"2022-01-26 13:15:23,708 [DQC.US.0081.9278] The...",2022-01-26,ERROR,DQC,81,1458581,"Zenosense, Inc.",2022-01-26 12:58:00,0001829126-22-001655,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_06B_USD_1_iso42...,10-K,http://www.sec.gov/Archives/edgar/data/1458581...,2022-01-26,2022,413859,https://www.sec.gov/Archives/edgar/data/145858...,1040,10k-zeno-20211231
1,DQC.US.0081.9278,"2024-11-19 06:16:38,077 [DQC.US.0081.9278] The...",2024-11-19,ERROR,DQC,81,1481504,"XERIANT, INC.",2024-11-14 15:36:00,0001477932-24-007198,US GAAP 2024,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1481504...,2024-11-14,2024,781689,https://www.sec.gov/Archives/edgar/data/148150...,3721,10q-xeri-20240930
2,DQC.US.0081.9278,"2022-02-09 13:00:29,657 [DQC.US.0081.9278] The...",2022-02-09,ERROR,DQC,81,55242,KENNAMETAL INC.,2022-02-09 12:31:00,0000055242-22-000007,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/55242/0...,2022-02-09,2022,417189,https://www.sec.gov/Archives/edgar/data/55242/...,3541,10q-kmt-20211231
3,DQC.US.0081.9278,"2022-11-21 17:48:29,643 [DQC.US.0081.9278] The...",2022-11-21,ERROR,DQC,81,1355677,Mexus Gold us,2022-11-21 16:08:00,0001096906-22-002820,US GAAP 2022,"Produced by EDGARsuite software, Advanced Comp...",10-Q,http://www.sec.gov/Archives/edgar/data/1355677...,2022-11-21,2022,531019,https://www.sec.gov/Archives/edgar/data/135567...,1040,10q-mxsg-20220930
4,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_058_edei%2D%2DD...,10-Q,http://www.sec.gov/Archives/edgar/data/1350156...,2022-01-14,2022,412124,https://www.sec.gov/Archives/edgar/data/135015...,5700,10q-paxh-20211130
5,DQC.US.0081.9278,"2023-11-17 13:16:23,434 [DQC.US.0081.9278] The...",2023-11-17,ERROR,DQC,81,1450307,"CORPORATE UNIVERSE, INC.",2023-11-17 12:40:00,0001654954-23-014526,US GAAP 2023,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1450307...,2023-11-17,2023,656231,https://www.sec.gov/Archives/edgar/data/145030...,3690,10q-couv-20230930
6,DQC.US.0081.9872,"2023-02-08 15:30:36,217 [DQC.US.0081.9872] The...",2023-02-08,ERROR,DQC,81,875729,"BION ENVIRONMENTAL TECHNOLOGIES, INC.",2023-02-08 14:53:00,0001079973-23-000168,US GAAP 2022,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/875729/...,2023-02-08,2023,546586,https://www.sec.gov/Archives/edgar/data/875729...,2870,10q-bnet-20221231
7,DQC.US.0081.9277,"2022-11-18 17:30:32,016 [DQC.US.0081.9277] The...",2022-11-18,ERROR,DQC,81,1084133,REAL BRANDS INC.,2022-11-18 16:43:00,0001264931-22-000192,US GAAP 2022,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/1084133...,2022-11-18,2022,530751,https://www.sec.gov/Archives/edgar/data/108413...,2080,10q-rlbd-20220930
8,DQC.US.0081.9278,"2022-11-21 17:48:29,644 [DQC.US.0081.9278] The...",2022-11-21,ERROR,DQC,81,1355677,Mexus Gold us,2022-11-21 16:08:00,0001096906-22-002820,US GAAP 2022,"Produced by EDGARsuite software, Advanced Comp...",10-Q,http://www.sec.gov/Archives/edgar/data/1355677...,2022-11-21,2022,531019,https://www.sec.gov/Archives/edgar/data/135567...,1040,10q-mxsg-20220930
9,DQC.US.0081.9277,"2022-01-18 07:01:54,631 [DQC.US.0081.9277] The...",2022-01-18,ERROR,DQC,81,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,US GAAP 2020,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1796949...,2022-01-18,2022,412314,https://www.sec.gov/Archives/edgar/data/179694..

## LLM judge assisstant

In [35]:
Client = OpenAI()

In [36]:
re_81_system_prompt = """You are given a DQC validation message. Your task is to extract two specific pieces of information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "00000002 - Statement - BALANCE SHEET", ignore any prefixes such as ID numbers or "- Statement -". Only return the clean name of the statement (e.g., "BALANCE SHEET").

2. **Erroneous us-gaap Tag Relationships**: From all `us-gaap:` tags mentioned in the message, identify pairs of tags where the relationship between them is described as incorrect or inconsistent. A tag pair should be extracted **only if** it satisfies both of the following conditions:
   - The two tags are stated to be in a specific structural relationship in the filer’s taxonomy (e.g., sibling, parent-child).
   - The message clearly indicates that this relationship contradicts the definition in the official US GAAP taxonomy, or requires reviewer attention for potential misclassification.

3. **Reasoning**: For each extracted tag pair, provide a short explanation summarizing the inconsistency or incorrect relationship between them as described in the message.

Output your result as a structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": [
    {
      "tag1": "...",
      "tag2": "...",
      "reason": "..."
    }
  ]
}
"""

In [37]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": re_81_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [38]:
res = []
for _, row in tqdm(re_81_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

10it [00:30,  3.06s/it]


In [39]:
res[6]

{'statement_name': 'CONSOLIDATED STATEMENTS OF OPERATIONS (UNAUDITED)',
 'error_tags': [{'tag1': 'us-gaap:NetIncomeLossAttributableToNoncontrollingInterest',
   'tag2': 'us-gaap:NetIncomeLoss',
   'reason': 'In the extension taxonomy, these tags are represented as siblings, but according to the US GAAP taxonomy calculation relationships, us-gaap:NetIncomeLossAttributableToNoncontrollingInterest is a descendant of us-gaap:NetIncomeLoss, indicating an incorrect structural relationship.'}],
 'folder_name': '10q-bnet-20221231',
 'segmentation_name': 'ConsolidatedStatementsOfOperations'}

In [40]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    the options of rule_id are 0081, 0001, and 0145
"""
new_re_81_df = construct_annotation(re_81_df, res, task_name="relationship_extraction", rule_id="0081")

In [41]:
new_re_81_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0081.9278,"2022-01-26 13:15:23,708 [DQC.US.0081.9278] The...",2022-01-26,ERROR,DQC,81,1458581,"Zenosense, Inc.",2022-01-26 12:58:00,0001829126-22-001655,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:DueToOtherRelatedPartiesClassifiedCu...,10k-zeno-20211231/BalanceSheet,True,"{'statement_name': 'BALANCE SHEET', 'error_tag..."
1,DQC.US.0081.9278,"2024-11-19 06:16:38,077 [DQC.US.0081.9278] The...",2024-11-19,ERROR,DQC,81,1481504,"XERIANT, INC.",2024-11-14 15:36:00,0001477932-24-007198,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:DepositsAssetsCurrent, Reversal, us-...",10q-xeri-20240930/CondensedConsolidatedBalance...,True,{'statement_name': 'CONDENSED CONSOLIDATED BAL...
2,DQC.US.0081.9278,"2022-02-09 13:00:29,657 [DQC.US.0081.9278] The...",2022-02-09,ERROR,DQC,81,55242,KENNAMETAL INC.,2022-02-09 12:31:00,0000055242-22-000007,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:OtherComprehensiveIncomeLossCashFlow...,10q-kmt-20211231/CONDENSEDCONSOLIDATEDSTATEMEN...,True,{'statement_name': 'CONDENSED CONSOLIDATED STA...
3,DQC.US.0081.9278,"2022-11-21 17:48:29,643 [DQC.US.0081.9278] The...",2022-11-21,ERROR,DQC,81,1355677,Mexus Gold us,2022-11-21 16:08:00,0001096906-22-002820,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<linkb...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:OtherAssetsNoncurrent, Reversal, us-...",10q-mxsg-20220930/idr_StatementCondensedConsol...,True,{'statement_name': 'Condensed Consolidated Bal...
4,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:AccountsPayableRelatedPartiesCurrent...,10q-paxh-20211130/CondensedConsolidatedBalance...,True,{'statement_name': 'CONDENSED CONSOLIDATED BAL...
5,DQC.US.0081.9278,"2023-11-17 13:16:23,434 [DQC.US.0081.9278] The...",2023-11-17,ERROR,DQC,81,1450307,"CORPORATE UNIVERSE, INC.",2023-11-17 12:40:00,0001654954-23-014526,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:OtherAssetsNoncurrent, Reversal, us-...",10q-couv-20230930/ConsolidatedBalanceSheets,True,{'statement_name': 'CONSOLIDATED BALANCE SHEET...
6,DQC.US.0081.9872,"2023-02-08 15:30:36,217 [DQC.US.0081.9872] The...",2023-02-08,ERROR,DQC,81,875729,"BION ENVIRONMENTAL TECHNOLOGIES, INC.",2023-02-08 14:53:00,0001079973-23-000168,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:NetIncomeLossAttributableToNoncontro...,10q-bnet-20221231/ConsolidatedStatementsOfOper...,True,{'statement_name': 'CONS

In [42]:
new_re_81_df.to_excel(output_data,index=False)